# 06 — Grounded Product QA Demo

Demonstrate the production QA pipeline on real product comments.

The answer generator is constrained to retrieved evidence and every citation ID
is validated before the response is returned.

In [ ]:
from pathlib import Path
import sys
import os
import json

from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.rag.config import load_config
from src.rag.generation import OpenAIJSONGenerator
from src.rag.pipeline import GroundedQAPipeline
from src.rag.runtime import load_retrieval_stack

load_dotenv(PROJECT_ROOT / ".env")

In [ ]:
qa_config = load_config(
    PROJECT_ROOT / "configs" / "qa.yaml"
)

stack = load_retrieval_stack(PROJECT_ROOT)

generator = OpenAIJSONGenerator(
    api_key=os.getenv("METIS_API_KEY"),
    base_url=os.getenv("METIS_BASE_URL"),
    model=qa_config["generation"]["model"],
    input_cost_per_million=qa_config[
        "generation"
    ].get("input_cost_per_million"),
    output_cost_per_million=qa_config[
        "generation"
    ].get("output_cost_per_million"),
)

qa_pipeline = GroundedQAPipeline(
    retriever=stack.hybrid,
    generator=generator,
    documents=stack.documents,
    top_k=qa_config["qa"]["top_k"],
    max_context_chars=qa_config[
        "qa"
    ]["max_context_chars"],
    max_chars_per_comment=qa_config[
        "qa"
    ]["max_chars_per_comment"],
)

## Benchmark sample

In [ ]:
EVAL_PATH = (
    PROJECT_ROOT
    / "data"
    / "evaluation"
    / "retrieval_queries.json"
)

with open(EVAL_PATH, encoding="utf-8") as file:
    samples = json.load(file)

sample = samples[0]

print("Product:", sample["product_title"])
print("Query:", sample["query"])

result = qa_pipeline.answer(
    query=sample["query"],
    product_id=sample["product_id"],
)

print("\nANSWER")
print(result["answer"])
print("\nEvidence IDs:", result["evidence_ids"])
print("Confidence:", result["confidence"])
print("Citation valid:", result["citation_valid"])
print("Insufficient evidence:", result["insufficient_evidence"])

In [ ]:
display(
    result["evidence_documents"][
        ["id", "rate", "body", "score"]
    ]
)

result["telemetry"]

## Custom question

In [ ]:
custom_result = qa_pipeline.answer(
    query="ایرادهای پرتکرار این محصول چیست؟",
    product_id=sample["product_id"],
)

print(custom_result["answer"])

display(
    custom_result["evidence_documents"][
        ["id", "rate", "body", "score"]
    ]
)

custom_result["telemetry"]